# Módulo 14: Tipos de analítica — Guía del ejercicio

En este ejercicio vas a distinguir los cuatro tipos de analítica (descriptiva, diagnóstica, predictiva y prescriptiva) y los roles que suelen responder a cada uno: analista de datos, científico de datos e ingeniero de datos. Primero clasificarás preguntas de negocio de una tienda en línea. Después responderás en Python una pregunta de cada tipo con los datos de ventas de la tienda.

- Si necesitas repasar los roles, sus responsabilidades, herramientas y aptitudes, consulta la lección [m14-tipos-de-analitica-leccion.md](m14-tipos-de-analitica-leccion.md).
- Cuando termines, compara tu trabajo con la solución: [m14-tipos-de-analitica-solucion.ipynb](m14-tipos-de-analitica-solucion.ipynb).

## Contexto

**Casa Ámbar** es una tienda en línea de artículos para el hogar que vende y envía a cuatro regiones: Centro, Norte, Occidente y Sur. Hasta ahora, su equipo de datos se limitaba a una hoja de cálculo con las ventas del mes. Al cerrar 2025, la dirección quiere usar mejor sus datos y plantea muchas preguntas a la vez: cuánto se vendió, por qué una de las regiones bajó en el último trimestre, cuánto se venderá en enero y si conviene lanzar un descuento.

Antes de contratar personal, la dirección te pide ordenar esas preguntas: qué **tipo de analítica** requiere cada una y qué **rol** debería responderla. Después te pide responder una pregunta de cada tipo con los datos diarios de ventas de 2025.

Recuerda los cuatro tipos de analítica:

| Tipo | Pregunta que responde |
| --- | --- |
| Descriptiva | ¿Qué pasó? |
| Diagnóstica | ¿Por qué pasó? |
| Predictiva | ¿Qué es probable que pase? |
| Prescriptiva | ¿Qué deberíamos hacer? |

## Datos

La celda inicial genera el DataFrame `ventas`, con una fila por día y región del 1 de enero al 31 de diciembre de 2025:

| Columna | Descripción |
| --- | --- |
| `fecha` | Día |
| `region` | Región de envío: `Centro`, `Norte`, `Occidente` o `Sur` |
| `visitas` | Visitas al sitio desde esa región |
| `pedidos` | Pedidos completados |
| `ingresos` | Ingresos del día en pesos (precio de lista, sin descuentos) |
| `dias_entrega` | Tiempo promedio de entrega de los pedidos del día, en días |

## Instrucciones

- Las tareas 1 y 2 son preguntas escritas. Respóndelas en la celda de texto que sigue a cada una, con tus palabras y apoyándote en la lección.
- Las tareas 3 a 6 son la práctica en Python, una por cada tipo de analítica. Escribe tu código en las celdas vacías y añade más celdas si lo necesitas.
- Acompaña cada resultado con una frase breve que lo interprete.
- La celda siguiente importa las librerías y simula los datos de la tienda con una semilla fija. Trátala como la base de datos de la empresa: no la modifiques y descubre los patrones con el análisis, no leyendo el código que los genera.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

fechas = pd.date_range("2025-01-01", "2025-12-31", freq="D")
regiones = ["Centro", "Norte", "Occidente", "Sur"]
visitas_base = np.array([5200, 3400, 3000, 2600])
entrega_base = np.array([2.6, 3.1, 3.3, 2.9])
entrega_extra = rng.permutation([0.1, 0.0, 3.0, 0.2])
mes_cambio = rng.integers(8, 11)

t = (fechas - fechas[0]).days.to_numpy()
dia_semana = np.array([1.05, 1.0, 0.98, 0.97, 1.0, 0.92, 1.08])[fechas.dayofweek]
partes = []
for i, region in enumerate(regiones):
    visitas = rng.poisson(visitas_base[i] * (1 + 0.0006 * t) * dia_semana)
    cambio = (fechas.month >= mes_cambio) * entrega_extra[i]
    dias_entrega = np.round(entrega_base[i] + cambio + rng.normal(0, 0.3, len(fechas)), 1).clip(1)
    conversion = 0.021 * np.exp(-0.12 * (dias_entrega - 3)) * rng.normal(1, 0.06, len(fechas))
    pedidos = rng.binomial(visitas, conversion.clip(0.001, 0.1))
    ticket = rng.normal(820, 45, len(fechas))
    partes.append(pd.DataFrame({
        "fecha": fechas,
        "region": region,
        "visitas": visitas,
        "pedidos": pedidos,
        "ingresos": np.round(pedidos * ticket, 2),
        "dias_entrega": dias_entrega,
    }))
ventas = pd.concat(partes, ignore_index=True)

print(ventas.shape)
ventas.head()

## Tarea 1. Tipo de analítica de cada pregunta

Estas son las ocho preguntas que plantea la dirección:

1. ¿Cuántos ingresos tendremos en enero de 2026?
2. ¿Cuántos ingresos, pedidos y visitas tuvimos cada mes de 2025?
3. ¿Qué porcentaje de descuento conviene ofrecer en enero para obtener el mayor margen?
4. ¿Por qué bajaron las ventas de una de las regiones en el último trimestre?
5. ¿Cuántos pedidos se están registrando en la última hora en cada región?
6. ¿Qué clientes tienen alta probabilidad de no volver a comprar en los próximos 90 días?
7. ¿Las devoluciones aumentaron porque cambiamos de proveedor de textiles?
8. ¿Qué paquetería conviene asignar a cada región para reducir el costo de envío sin pasar de 4 días de entrega?

Para cada pregunta, indica si es **descriptiva**, **diagnóstica**, **predictiva** o **prescriptiva** y justifica en una frase. Presenta tu respuesta en una tabla.

*Escribe aquí tu respuesta.*

## Tarea 2. Rol responsable de cada pregunta

1. Añade a tu tabla de la Tarea 1 el **rol** que debería encabezar la respuesta de cada pregunta: analista de datos, científico de datos o ingeniero de datos. Justifica cada elección con las responsabilidades y herramientas que describe la lección.
2. Varias preguntas necesitan datos que hoy no se recogen o que llegan tarde (por ejemplo, la 5, la 6 y la 7). Explica qué tendría que construir el **ingeniero de datos** para que se puedan responder.
3. Si Casa Ámbar solo pudiera contratar a una persona este año, ¿qué rol elegirías y por qué? Considera también las aptitudes académicas y personales de cada rol.

*Escribe aquí tu respuesta.*

## Tarea 3. Analítica descriptiva: KPIs mensuales

Responde la pregunta 2: ¿qué pasó en 2025?

1. Agrega `ventas` por mes (para toda la tienda) y calcula: ingresos, pedidos, visitas, **ticket promedio** (ingresos / pedidos) y **tasa de conversión** (pedidos / visitas, en porcentaje).
2. Muestra la tabla de 12 meses con valores redondeados.
3. Grafica los ingresos mensuales e indica qué meses tuvieron los ingresos más altos y más bajos.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 4. Analítica diagnóstica: ¿por qué cayó una región?

Responde la pregunta 4.

1. Construye una tabla de ingresos mensuales por región y grafícala con una línea por región. Identifica **qué región** cayó y **desde qué mes**.
2. Para cada región, compara el periodo anterior a la caída con el periodo posterior en cuatro variables: visitas diarias promedio, tasa de conversión, ticket promedio y días de entrega promedio. Presenta las diferencias en una tabla.
3. Calcula, con los datos mensuales por región, la correlación entre los días de entrega promedio y la tasa de conversión.
4. Explica cuál es la causa más probable de la caída y por qué descartas las demás variables. ¿Qué dato adicional pedirías para confirmarlo?

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 5. Analítica predictiva: pronóstico de enero de 2026

Responde la pregunta 1. Como los meses tienen distinto número de días, trabaja con el **ingreso diario promedio** de cada mes.

1. Calcula el ingreso diario promedio de cada mes de 2025 (ingresos del mes / días del mes) y numera los meses del 1 al 12.
2. Ajusta una recta (tendencia lineal) del ingreso diario promedio en función del número de mes, con `np.polyfit` o con `LinearRegression` de scikit-learn. Reporta la pendiente e interprétala.
3. Pronostica el ingreso diario promedio del mes 13 (enero de 2026) y multiplícalo por 31 días para obtener los ingresos de enero.
4. Grafica los datos, la recta y el pronóstico. Menciona al menos una limitación de este pronóstico.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 6. Analítica prescriptiva: el descuento que maximiza el margen

Responde la pregunta 3. La dirección quiere ofrecer en enero un descuento $d$ sobre el precio de lista y te da estos supuestos:

- Sin descuento, la demanda de enero sería igual a los **pedidos de diciembre de 2025**, $Q_0$, con un precio igual al **ticket promedio de diciembre**, $p_0$.
- El costo de cada pedido es el **60 %** del precio de lista: $c = 0.6\,p_0$.
- La demanda responde al precio con una **elasticidad constante** $\varepsilon = 3$ (en valor absoluto): una rebaja del 1 % en el precio aumenta los pedidos alrededor de un 3 %.

$$Q(d) = Q_0\,(1 - d)^{-\varepsilon} \qquad \text{margen}(d) = Q(d)\,\bigl(p_0\,(1 - d) - c\bigr)$$

1. Calcula $Q_0$ y $p_0$ a partir de `ventas`.
2. Simula el margen para los descuentos de 0 % a 30 % en pasos de 1 %. Muestra una tabla con el descuento, los pedidos, los ingresos y el margen para 0 %, 5 %, 10 %, 15 %, 20 % y 30 %.
3. Encuentra el descuento que maximiza el margen y cuánto margen adicional genera frente a no hacer descuento. Grafica el margen en función del descuento.
4. Repite la simulación con $\varepsilon = 2$. ¿Cambia la recomendación? Explica por qué y redacta la recomendación para la dirección.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Antes de terminar

- [ ] Clasificaste las ocho preguntas por tipo de analítica y por rol, con una justificación para cada una.
- [ ] Explicaste qué debe construir el ingeniero de datos y qué rol contratarías primero.
- [ ] Tu tabla de KPIs tiene 12 meses con ingresos, pedidos, visitas, ticket promedio y conversión.
- [ ] Identificaste la región que cayó, desde qué mes y la variable que mejor explica la caída.
- [ ] Tu pronóstico de enero de 2026 viene acompañado de la recta ajustada y de una limitación.
- [ ] Encontraste el descuento óptimo para $\varepsilon = 3$ y para $\varepsilon = 2$, y redactaste una recomendación.

Cuando termines, compara tu trabajo con la solución: [m14-tipos-de-analitica-solucion.ipynb](m14-tipos-de-analitica-solucion.ipynb).